# Hotel booking data pipeline

**Bronze → Silver → Gold.** This notebook is the project guide and execution entry point.
It prepares separate H1 and H2 tables, with one row per source booking.

**Read only:** browse this notebook to understand the supplied data.  
**Rebuild:** start the course environment, then run the Python cells from top to bottom.
Rebuilding overwrites derived tables; original downloads remain unchanged.

This reorganized notebook has not been executed. The counts below describe the existing supplied Gold files.

## 1. Data flow and files

![Hotel data pipeline](diagrams/Hotel_Data_Pipeline_T14.png)

| Location | Contents |
| --- | --- |
| `README.ipynb` | Guide, processing rules and execution steps. |
| `AGENTS.md` | Data-processing conventions for code changes. |
| `data/` | Original H1/H2 files, external downloads and source records. |
| `utils/` | Assignment 1-style processing functions. |
| `datamart/` | Bronze CSVs; Silver and Gold Parquet tables. |
| `EDA/` | One [Alex notebook](EDA/Alex_EDA_Hotel_Booking.ipynb) and one [Helen notebook](EDA/Helen_EDA_Hotel_Booking.ipynb). |
| `diagrams/` | Latest data pipeline PNG. |

Transfer the whole folder, including `data/` and `datamart/`; `.gitignore` excludes the large data files.
The two EDA notebooks are historical references. Current processing functions and the rules below define the delivered data.

## 2. Inputs and delivered Gold

| Source | Location / use |
| --- | --- |
| H1 Resort: 40,060 bookings | `data/raw/H1.csv` |
| H2 City: 79,330 bookings | `data/raw/H2.csv` |
| Portugal national holidays | `data/external/holidays/`; join by arrival date. Correct four suspended holidays in 2015. |
| EUR/USD | `data/external/exchange_rates/`; USD per EUR, latest observation strictly before `CutoffDate`. |
| Google Trends | `data/external/google_trends/`; fixed worldwide four-term weekly export. H1: Algarve hotels / flights to Faro. H2: Lisbon hotels / flights to Lisbon. |

[Source records](data/external/source_manifest.json) contain download URLs and hashes. Keep original snapshots unchanged.
For Trends, discard partial boundary weeks and select the latest week where **week end + 7 days ≤ CutoffDate**.
The seven-day buffer is an assumption. The retrospective 0–100 export is not an original historical snapshot, and different export scales are not directly comparable.

| Hotel | Gold rows | IsCanceled = 1 | Columns |
| --- | ---: | ---: | ---: |
| H1 | 20,140 | 1,518 (7.54%) | 41 |
| H2 | 38,233 | 4,536 (11.86%) | 41 |

**Parquet:** `datamart/gold/model_dataset/gold_model_dataset_H1.parquet/` and `gold_model_dataset_H2.parquet/`.
Read a complete hotel directory, not an individual part file. The directory naming follows Assignment 1.

**Excel references:** [H1](datamart/gold/model_dataset/excel_reference/gold_model_dataset_H1.xlsx) / [H2](datamart/gold/model_dataset/excel_reference/gold_model_dataset_H2.xlsx).
These static copies are not refreshed by this notebook.

## 3. Processing rules

| Item | Rule |
| --- | --- |
| Source records | Keep identical-looking rows. Create `Booking_ID` from hotel + original CSV row number. Preserve source order. |
| Missing values | Trim strings; `NULL` / `NA` / empty values become missing. Agent/Company `NULL` means no recorded agent/company. |
| Dates | Combine arrival year/month/day internally; derive `ArrivalDayOfWeek` (Monday=1). `BookingCreatedDate = ArrivalDate − LeadTime`. |
| Guests and nights | `PartySize = Adults + Children + Babies`; `TotalNights = StaysInWeekNights + StaysInWeekendNights`. Preserve missing inputs. |
| ADR | Negative → missing; zero is kept. Values above 1,000 are flagged for review, not automatically removed. |
| Date differences | Keep and flag the 28 identified source rows; do not invent corrected dates. |
| Estimated rooms | `ceil(PartySize / 2)` for H1; `ceil(PartySize / 4)` for H2. Assumed reporting quantity, not observed rooms. |
| Categories | Preserve text and documented missing values in Gold. One H1 ADR is missing. |

**T-14 means 14 days before scheduled arrival.** `CutoffDate = ArrivalDate − 14 days`.
Gold includes bookings with `LeadTime ≥ 14`, positive `PartySize` and positive `TotalNights`.
Exclude bookings with `ReservationStatus` in Canceled / No-Show and `ReservationStatusDate ≤ CutoffDate`.
The boundary is **end of day**: include bookings created on T-14; exclude cancellations/no-shows recorded that day.

Retained booking details are assumed unchanged before the cutoff. The source does not provide daily booking snapshots.
Silver preserves all source rows; Gold applies the agreed scope filters.

**Gold columns:** 30 prepared features + 10 metadata fields + existing `IsCanceled`.
The [feature contract](datamart/gold/feature_contract.json) lists the exact groups.
`IsCanceled = 1` means later cancellation/no-show; `0` means completed stay.

The feature group excludes `Country`, `AssignedRoomType`, `RequiredCarParkingSpaces`, `Agent`, `Company`, `ReservationStatus` and `ReservationStatusDate`, following the agreed scope.
IDs, full dates and estimated rooms remain metadata. Final status/date are kept in `datamart/gold/labels/` for audit.
Silver review records are in `datamart/silver/hotels/silver_review_H1.parquet/` and the corresponding H2 directory.

## 4. Start the course environment

Use the unchanged Assignment 1 Docker files. Run these commands in a terminal opened at this project folder:

```sh
docker compose build
docker compose up -d
docker compose logs jupyter
```

Open `http://localhost:8888`, then open **README.ipynb** and select the Python kernel.
The project is mounted at `/app`; the container name is `jupyter_lab`.
If another course container uses the same name/port, stop it first or use an environment with this project mounted.
The course setup has no token authentication; keep it local.

**Execution order:** initialize → Bronze → Silver → Gold → check saved outputs → stop Spark.
All required downloads are supplied. No extra entry-point script is needed.

### 4.1 Initialize Spark and paths
The processing modules keep Assignment 1's simple read → transform → select → save structure.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/raw/H1.csv').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import pyspark
from IPython.display import display
import pandas as pd
import pyspark.sql.functions as F
import utils.data_processing_bronze_table as bronze_table
import utils.data_processing_bronze_features as bronze_features
import utils.data_processing_silver_table as silver_table
import utils.data_processing_silver_features as silver_features
import utils.data_processing_gold_features as gold_features
import utils.data_processing_gold_table as gold_table
import utils.data_processing_gold_model_dataset as gold_model

# Initialize SparkSession; two workers keep this laptop run bounded.
spark = pyspark.sql.SparkSession.builder.appName('hotel_booking_pipeline').master('local[2]').config('spark.sql.shuffle.partitions','4').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark:',spark.version)
bronze_hotels = 'datamart/bronze/hotels/'
bronze_external = 'datamart/bronze/external/'
silver_hotels = 'datamart/silver/hotels/'
silver_external = 'datamart/silver/external/'
silver_scoring = 'datamart/silver/scoring/'
gold_feature_store = 'datamart/gold/features/'
gold_label_store = 'datamart/gold/labels/'
gold_model_dataset = 'datamart/gold/model_dataset/'

### 4.2 Bronze — preserve source values
Ingest H1/H2 and the three external sources. Save CSV tables under `datamart/bronze/`.

In [ ]:
for hotel in ['H1','H2']:
    bronze_table.process_bronze_table(hotel,'data/raw/',bronze_hotels,spark)
bronze_features.process_bronze_holidays('data/external/holidays/',bronze_external,spark)
bronze_features.process_bronze_exchange_rates('data/external/exchange_rates/',bronze_external,spark)
bronze_features.process_bronze_google_trends('data/external/google_trends/',bronze_external,spark)
print('Bronze completed.')

### 4.3 Silver — clean and derive fields
Apply explicit types, documented cleaning, booking dates, guests, nights and estimated rooms.
Prepare the external records available by each hotel's cutoff date. Save Parquet under `datamart/silver/`.

In [ ]:
for hotel in ['H1','H2']:
    silver_table.process_silver_table(hotel,bronze_hotels,silver_hotels,spark)
silver_features.process_silver_holidays(bronze_external,silver_external,spark)
silver_features.process_silver_exchange_rates(bronze_external,silver_external,spark)
silver_features.process_silver_google_trends(bronze_external,silver_external,spark)
for hotel in ['H1','H2']:
    silver_features.process_silver_scoring_google_trends(hotel,silver_hotels,silver_external,silver_scoring,spark)
    silver_features.process_silver_scoring_exchange_rates(hotel,silver_hotels,silver_external,silver_scoring,spark)
print('Silver completed.')

### 4.4 Gold — filter and combine
Apply T-14 and overnight-stay scope, join external features, and combine with the existing label by `Booking_ID`.
Save separate H1/H2 feature, label and final joined tables under `datamart/gold/`.

In [ ]:
for hotel in ['H1','H2']:
    gold_features.process_features_gold_table(hotel,silver_hotels,silver_external,silver_scoring,gold_feature_store,spark)
    gold_table.process_labels_gold_table(hotel,silver_hotels,gold_label_store,spark)
    gold_model.process_model_dataset_gold_table(hotel,gold_feature_store,gold_label_store,gold_model_dataset,spark)
print('Gold completed.')

### 4.5 Check saved outputs
Check row counts, unique booking IDs, column groups, label values and basic scope conditions.
Expected counts apply to the supplied fixed source files. This is a compact structural check; it does not independently recalculate every cleaning and external join.

In [ ]:
expected = {'H1': (20140, 1518), 'H2': (38233, 4536)}
expected_columns = (gold_features.METADATA_COLUMNS + gold_features.NUMERIC_COLUMNS
                    + gold_features.CATEGORICAL_COLUMNS + ['IsCanceled'])
summary = []

for hotel in ['H1', 'H2']:
    filepath = os.path.join(gold_model_dataset, 'gold_model_dataset_' + hotel + '.parquet')
    df = spark.read.parquet(filepath)
    counts = df.agg(
        F.count('*').alias('Rows'),
        F.countDistinct('Booking_ID').alias('UniqueBookings'),
        F.sum(F.col('IsCanceled')).alias('CanceledOrNoShow'),
    ).first()
    assert counts['Rows'] == counts['UniqueBookings'], hotel + ': duplicate or missing Booking_ID'
    assert (counts['Rows'], counts['CanceledOrNoShow']) == expected[hotel], hotel + ': unexpected counts'
    assert len(df.columns) == len(expected_columns) and set(df.columns) == set(expected_columns)
    assert df.filter(F.col('IsCanceled').isNull() | ~F.col('IsCanceled').isin(0, 1)).limit(1).count() == 0

    in_scope = ((F.col('LeadTime') >= 14) & (F.col('PartySize') > 0)
                & (F.col('TotalNights') > 0)
                & (F.col('CutoffDate') == F.date_sub(F.col('ArrivalDate'), 14)))
    assert df.filter(~F.coalesce(in_scope, F.lit(False))).limit(1).count() == 0
    summary.append({'Hotel': hotel, 'Rows': counts['Rows'], 'Columns': len(df.columns),
                    'IsCanceled = 1': counts['CanceledOrNoShow']})

display(pd.DataFrame(summary))

### 4.6 Finish
Stop Spark after reviewing the saved outputs. Existing Excel references remain unchanged.

In [ ]:
spark.stop()